# 02 - Product Data Pipeline

Notebook này dùng để **chuẩn hóa dữ liệu sản phẩm và index vào Qdrant**. Phần triển khai dài được thu gọn mặc định; các cell thực thi vẫn giữ output cần thiết cho debug.

### 02 - Product Data Pipeline

- **Tác dụng chính:** Notebook này dùng để chuẩn hóa dữ liệu sản phẩm và index vào Qdrant. Phần triển khai dài được thu gọn mặc định; các cell thực thi vẫn giữ output cần thiết cho debug.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `resolve_chatbot_fashion_dir`, `CHATBOT_FASHION_DIR`, `METADATA_FILE` cùng output debug như shape, ID, score hoặc trạng thái trung gian.


In [ ]:
from pathlib import Path

import json

import random

from langchain_core.documents import Document

def resolve_chatbot_fashion_dir() -> Path:
    """Tìm thư mục gốc `Chatbot_Fashion/` nếu notebook 01 chưa được chạy trong kernel này.

    Args:
        Không có.

    Returns:
        Path: Dữ liệu đã được chuẩn hóa hoặc suy ra để dùng ở bước tiếp theo.
    """
    cwd = Path.cwd()
    if cwd.name == "notebooks" and cwd.parent.name == "Chatbot_Fashion":
        return cwd.parent
    if cwd.name == "Chatbot_Fashion":
        return cwd
    candidate = cwd / "Chatbot_Fashion"
    if candidate.exists():
        return candidate
    for parent in cwd.parents:
        candidate = parent / "Chatbot_Fashion"
        if candidate.exists():
            return candidate
    raise FileNotFoundError("Không tìm thấy thư mục Chatbot_Fashion/")


# Nếu notebook 01 đã chạy, biến CHATBOT_FASHION_DIR có thể đã tồn tại.
# Nếu chưa, cell này tự resolve để notebook 02 vẫn đọc được dữ liệu.


In [1]:
CHATBOT_FASHION_DIR = globals().get("CHATBOT_FASHION_DIR", resolve_chatbot_fashion_dir())

METADATA_FILE = CHATBOT_FASHION_DIR / "data" / "metadata" / "meta_Amazon_Lazada_Fashion_65k.jsonl"

print(f"[OK] Project root  : {CHATBOT_FASHION_DIR}")

print(f"[OK] Metadata file : {METADATA_FILE}")

print(f"     Exists        : {METADATA_FILE.exists()}")


[OK] Project root  : d:\KHÓA LUẬN\WORKSPACE\Chatbot_Fashion
[OK] Metadata file : d:\KHÓA LUẬN\WORKSPACE\Chatbot_Fashion\data\metadata\meta_Amazon_Lazada_Fashion_65k.jsonl
     Exists        : True


### BƯỚC 1: Tư Duy Data Pipeline - Biến Product Thô Thành Document

- **Tác dụng chính:** Thực hiện bước chuẩn hóa dữ liệu sản phẩm và index vào Qdrant.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `extract_image_urls` để các bước sau tiếp tục sử dụng.


In [2]:
def extract_image_urls(item: dict) -> list[str]:
    """Lấy tất cả URL/path ảnh chất lượng cao từ một product item.

    Args:
        item (dict): Bản ghi sản phẩm hoặc đối tượng đang xử lý.

    Returns:
        list[str]: Dữ liệu đã được chuẩn hóa hoặc suy ra để dùng ở bước tiếp theo.
    """
    images = item.get("images", []) or []
    return [img.get("large") for img in images if isinstance(img, dict) and img.get("large")]


### Bước 3: Chuẩn hóa dữ liệu sản phẩm và index vào qdrant

- **Tác dụng chính:** Thực hiện bước chuẩn hóa dữ liệu sản phẩm và index vào Qdrant.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `normalize_to_text` để các bước sau tiếp tục sử dụng.


In [3]:
def normalize_to_text(value, default: str = "Không rõ") -> str:
    """Chuẩn hóa một field bất kỳ thành chuỗi dùng được trong `page_content`.

    Args:
        value (Any): Giá trị đầu vào phục vụ bước xử lý này.
        default (str): Giá trị đầu vào phục vụ bước xử lý này.

    Returns:
        str: Dữ liệu đã được chuẩn hóa hoặc suy ra để dùng ở bước tiếp theo.
    """
    if value is None or value == "":
        return default

    if isinstance(value, list):
        cleaned = [str(item).strip() for item in value if str(item).strip()]
        return ", ".join(cleaned) if cleaned else default

    return str(value).strip() or default


### Bước 4: Chuẩn hóa dữ liệu sản phẩm và index vào qdrant

- **Tác dụng chính:** Thực hiện bước chuẩn hóa dữ liệu sản phẩm và index vào Qdrant.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `build_product_metadata` để các bước sau tiếp tục sử dụng.


In [4]:
def build_product_metadata(item: dict) -> dict:
    """Tạo metadata dict cho LangChain Document.

    Args:
        item (dict): Bản ghi sản phẩm hoặc đối tượng đang xử lý.

    Returns:
        dict: Dữ liệu đã được chuẩn hóa hoặc suy ra để dùng ở bước tiếp theo.
    """
    image_urls = extract_image_urls(item)
    return {
        "product_id": item.get("product_id", ""),
        "title": item.get("title", ""),
        "category": item.get("category", ""),
        "department": item.get("department", ""),
        "brand": item.get("brand", ""),
        "price": item.get("price", 0),
        "images": image_urls,
        "image_url": image_urls[0] if image_urls else "",
    }


### Bước 5: Chuẩn hóa dữ liệu sản phẩm và index vào qdrant

- **Tác dụng chính:** Thực hiện bước chuẩn hóa dữ liệu sản phẩm và index vào Qdrant.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `build_product_page_content` cùng output debug như shape, ID, score hoặc trạng thái trung gian.


In [ ]:
def build_product_page_content(item: dict) -> str:
    """Tạo text có nhãn rõ ràng để embed bằng ViFashionCLIP.

    Args:
        item (dict): Bản ghi sản phẩm hoặc đối tượng đang xử lý.

    Returns:
        str: Dữ liệu đã được chuẩn hóa hoặc suy ra để dùng ở bước tiếp theo.
    """
    details = item.get("details", {}) or {}
    fields = [
        ("Tên sản phẩm", item.get("title")),
        ("Mã sản phẩm", item.get("product_id")),
        ("Danh mục", item.get("category")),
        ("Đối tượng", item.get("department")),
        ("Thương hiệu", item.get("brand")),
        ("Giá", f"{item.get('price', 0)} VND"),
        ("Màu sắc", details.get("main_color")),
        ("Chất liệu", details.get("material")),
        ("Kích cỡ", details.get("size")),
        ("Họa tiết", details.get("pattern")),
        ("Mùa phù hợp", item.get("season")),
        ("Dịp sử dụng", item.get("occasion")),
        ("Mô tả", item.get("description")),
    ]
    return "\n".join(f"{label}: {normalize_to_text(value)}" for label, value in fields)


In [5]:
print("[OK] Hàm build Document đã sẵn sàng")


[OK] Hàm build Document đã sẵn sàng


### BƯỚC 2: Preview Một Sản Phẩm Trước Khi Index

- **Tác dụng chính:** Thực hiện bước chuẩn hóa dữ liệu sản phẩm và index vào Qdrant.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `iter_jsonl_items`, `preview_product_document`, `preview_doc` cùng output debug như shape, ID, score hoặc trạng thái trung gian.


In [ ]:
def iter_jsonl_items(file_path: str | Path, limit: int | None = None):
    """Yield valid JSONL product items.

    Args:
        file_path (str | Path): Đường dẫn dữ liệu hoặc tệp cần xử lý.
        limit (int | None): Giới hạn số phần tử được xử lý hoặc trả về.

    Returns:
        Any: Kết quả đã xử lý của hàm.
    """
    file_path = Path(file_path)
    count = 0
    with file_path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            try:
                yield json.loads(line)
                count += 1
                if limit is not None and count >= limit:
                    return
            except json.JSONDecodeError:
                continue

def preview_product_document(file_path: str | Path = METADATA_FILE, index: int | None = None) -> Document | None:
    """Preview one product after converting it to Document.

    Args:
        file_path (str | Path): Đường dẫn dữ liệu hoặc tệp cần xử lý.
        index (int | None): Giá trị đầu vào phục vụ bước xử lý này.

    Returns:
        Document | None: Kết quả đã xử lý của hàm.
    """
    selected_item = None
    selected_index = None

    if index is None:
        seen = 0
        for i, item in enumerate(iter_jsonl_items(file_path)):
            seen += 1
            if random.randrange(seen) == 0:
                selected_item = item
                selected_index = i
    else:
        for i, item in enumerate(iter_jsonl_items(file_path)):
            if i == index:
                selected_item = item
                selected_index = i
                break

    if selected_item is None:
        print(f"[ERROR] Could not select product from {file_path} with index={index}")
        return None

    doc = Document(
        page_content=build_product_page_content(selected_item),
        metadata=build_product_metadata(selected_item),
    )

    print(f"SELECTED INDEX: {selected_index}")
    print("RAW KEYS:")
    print(sorted(selected_item.keys()))
    print("\nMETADATA:")
    print(json.dumps(doc.metadata, ensure_ascii=False, indent=2)[:2000])
    print("\nPAGE_CONTENT:")
    print(doc.page_content[:2500])
    return doc

# Random product:


In [6]:
preview_doc = preview_product_document()


SELECTED INDEX: 50377
RAW KEYS:
['brand', 'category', 'department', 'description', 'details', 'images', 'occasion', 'price', 'product_id', 'season', 'title']

METADATA:
{
  "product_id": "pdp-i3246432557",
  "title": "Chân Váy Len A-Line Cạp Cao",
  "category": "Chân váy",
  "department": "Nữ",
  "brand": "La Chapelle",
  "price": 374400,
  "images": [
    "images/pdp-i3246432557_MAIN.jpg",
    "images/pdp-i3246432557_PT01.jpg",
    "images/pdp-i3246432557_PT02.jpg",
    "images/pdp-i3246432557_PT03.jpg",
    "images/pdp-i3246432557_PT04.jpg",
    "images/pdp-i3246432557_PT05.jpg"
  ],
  "image_url": "images/pdp-i3246432557_MAIN.jpg"
}

PAGE_CONTENT:
Tên sản phẩm: Chân Váy Len A-Line Cạp Cao
Mã sản phẩm: pdp-i3246432557
Danh mục: Chân váy
Đối tượng: Nữ
Thương hiệu: La Chapelle
Giá: 374400 VND
Màu sắc: Đen, Xám, Nâu
Chất liệu: Polyester
Kích cỡ: S (40-47kg), M (47.5-52.5kg), L (53-60kg), XL (60.5-70kg)
Họa tiết: Không rõ
Mùa phù hợp: Mùa thu, Mùa đông
Dịp sử dụng: Hàng ngày, Xã hội
Mô t

### BƯỚC 3: Đọc JSONL Và Index Vào Qdrant

- **Tác dụng chính:** Thực hiện bước chuẩn hóa dữ liệu sản phẩm và index vào Qdrant.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `process_fashion_metadata` cùng output debug như shape, ID, score hoặc trạng thái trung gian.


In [7]:
def process_fashion_metadata(file_path: str | Path) -> list[Document]:
    """Đọc file JSONL và chuyển từng dòng hợp lệ thành LangChain Document.

    Args:
        file_path (str | Path): Đường dẫn dữ liệu hoặc tệp cần xử lý.

    Returns:
        list[Document]: Kết quả đã xử lý của hàm.
    """
    file_path = Path(file_path)
    documents: list[Document] = []
    stats = {
        "total_lines": 0,
        "json_errors": 0,
        "missing_product_id": 0,
        "missing_category": 0,
        "missing_image": 0,
    }

    with file_path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue

            stats["total_lines"] += 1
            try:
                item = json.loads(line)
            except json.JSONDecodeError:
                stats["json_errors"] += 1
                continue

            metadata = build_product_metadata(item)
            if not metadata["product_id"]:
                stats["missing_product_id"] += 1
            if not metadata["category"]:
                stats["missing_category"] += 1
            if not metadata["image_url"]:
                stats["missing_image"] += 1

            documents.append(
                Document(
                    page_content=build_product_page_content(item),
                    metadata=metadata,
                )
            )

    print(
        "[THỐNG KÊ DATA] "
        f"dòng={stats['total_lines']} | docs={len(documents)} | "
        f"lỗi_json={stats['json_errors']} | "
        f"thiếu_id={stats['missing_product_id']} | "
        f"thiếu_category={stats['missing_category']} | "
        f"thiếu_ảnh={stats['missing_image']}"
    )
    return documents


### Bước 10: Chuẩn hóa dữ liệu sản phẩm và index vào qdrant

- **Tác dụng chính:** Thực hiện bước chuẩn hóa dữ liệu sản phẩm và index vào Qdrant.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `run_data_pipeline`, `collection_name`, `client` cùng output debug như shape, ID, score hoặc trạng thái trung gian.


In [ ]:
def run_data_pipeline(
    metadata_file: str | Path = METADATA_FILE,
    qdrant_url: str = "http://localhost:6333",
    collection_name: str = "fashion_products_vifashionclip_vi_65k_structured_vi",
    batch_size: int = 128,
):
    """End-to-end indexing Layer A text:

    Args:
        metadata_file (str | Path): Đường dẫn dữ liệu hoặc tệp cần xử lý.
        qdrant_url (str): Giá trị đầu vào phục vụ bước xử lý này.
        collection_name (str): Tên collection Qdrant đích.
        batch_size (int): Giới hạn số phần tử được xử lý hoặc trả về.

    Returns:
        Any: Kết quả đã xử lý của hàm.
    """
    from langchain_qdrant import QdrantVectorStore
    from qdrant_client import QdrantClient
    from qdrant_client.http.models import Distance, VectorParams

    metadata_file = Path(metadata_file)
    if not metadata_file.exists():
        print(f"[LỖI] Không tìm thấy metadata file: {metadata_file}")
        return

    if "ViFashionCLIPTextEmbeddings" not in globals():
        raise RuntimeError("Chưa có ViFashionCLIPTextEmbeddings. Hãy chạy notebook 01 trước.")

    print(f"[THÔNG BÁO] Đọc dữ liệu từ: {metadata_file}")
    all_docs = process_fashion_metadata(metadata_file)

    emb = ViFashionCLIPTextEmbeddings()
    client = QdrantClient(url=qdrant_url)

    if not client.collection_exists(collection_name):
        client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(size=512, distance=Distance.COSINE),
        )
        current_count = 0
        print(f"[OK] Đã tạo collection mới: {collection_name}")
    else:
        current_count = client.get_collection(collection_name).points_count
        print(f"[INFO] Collection đã có {current_count} sản phẩm.")

    remaining = all_docs[current_count:]
    if not remaining:
        print("[OK] Collection đã index đầy đủ, không cần làm thêm.")
        return

    vector_store = QdrantVectorStore(
        client=client,
        collection_name=collection_name,
        embedding=emb,
    )

    with tqdm(total=len(all_docs), initial=current_count, desc="Index Layer A", unit="SP") as progress:
        # Chia dữ liệu thành các batch nhỏ để giới hạn bộ nhớ khi index.
        for start in range(0, len(remaining), batch_size):
            batch_docs = remaining[start:start + batch_size]
            vector_store.add_documents(documents=batch_docs)
            progress.update(len(batch_docs))

    print("[OK] Index Layer A text hoàn tất.")


# Chỉ bỏ comment khi thật sự muốn index/re-index dữ liệu.
# run_data_pipeline()

from qdrant_client import QdrantClient

# Kết nối tới Qdrant local

collection_name = "fashion_products_vifashionclip_vi_65k_structured_vi"


In [9]:
print("[OK] Data pipeline đã sẵn sàng. Hãy preview dữ liệu trước khi index.")

client = QdrantClient(url="http://localhost:6333")

if client.collection_exists(collection_name):
    info = client.get_collection(collection_name)
    print(f"[OK] Collection '{collection_name}' tồn tại.")
    print(f"     Số lượng sản phẩm đã index: {info.points_count} points")
    print(f"     Trạng thái collection: {info.status}")
else:
    print(f"[CẢNH BÁO] Collection '{collection_name}' không tồn tại trong Qdrant!")


[OK] Data pipeline đã sẵn sàng. Hãy preview dữ liệu trước khi index.


[OK] Collection 'fashion_products_vifashionclip_vi_65k_structured_vi' tồn tại.
     Số lượng sản phẩm đã index: 65480 points
     Trạng thái collection: green
